# 03. 연구 인센티브의 toy model

목표: theorem-credit만 보상할 때와 설명·검증·연결을 함께 보상할 때 프로젝트 선택이 어떻게 달라지는지 탐색한다. 이는 사회과학적 증거가 아니라 논문의 주장을 명시적 가정으로 바꾸는 사고 실험이다.

## 1. 기여 유형과 보상

각 프로젝트는 정리, 설명, 검증, 분야 연결 점수를 가진다. 개인은 제도에서 높은 보상을 주는 프로젝트를 선택한다고 단순화한다.

In [ ]:
projects = {
    "new_theorem": {"theorem": 10, "explanation": 2, "verification": 3, "bridge": 1},
    "expository_notes": {"theorem": 1, "explanation": 10, "verification": 4, "bridge": 6},
    "proof_audit": {"theorem": 0, "explanation": 4, "verification": 10, "bridge": 2},
    "cross_field_workshop": {"theorem": 2, "explanation": 7, "verification": 2, "bridge": 10},
}

institutions = {
    "theorem_credit_only": {"theorem": 1.0, "explanation": 0.05, "verification": 0.05, "bridge": 0.05},
    "understanding_balanced": {"theorem": 0.4, "explanation": 0.3, "verification": 0.2, "bridge": 0.3},
}


def reward(project: dict, weights: dict) -> float:
    return sum(project[key] * weights[key] for key in weights)


for institution, weights in institutions.items():
    ranking = sorted(
        ((name, reward(values, weights)) for name, values in projects.items()),
        key=lambda row: row[1],
        reverse=True,
    )
    print(institution, ranking)

assert max(projects, key=lambda name: reward(projects[name], institutions["theorem_credit_only"])) == "new_theorem"

## 2. 공동체 portfolio

최고 점수 프로젝트 하나만 고르는 대신 예산 안에서 기여 유형의 coverage를 넓히는 간단한 조합을 찾는다.

In [ ]:
from itertools import combinations


def portfolio_value(names: tuple[str, ...]) -> tuple[int, int]:
    totals = {key: sum(projects[name][key] for name in names) for key in next(iter(projects.values()))}
    # 최소 기여가 큰 portfolio를 우선하고, 동률이면 총합을 사용한다.
    return min(totals.values()), sum(totals.values())


portfolios = list(combinations(projects, 2))
best = max(portfolios, key=portfolio_value)
print("balanced portfolio:", best, "value:", portfolio_value(best))
assert len(best) == 2

## 3. 모형의 한계와 실제 측정

점수와 가중치는 임의적이며 사람의 동기를 단일 reward maximization으로 축소했다. 실제 평가에서는 다음 자료가 필요하다.

- 설명 자료가 새 연구자의 진입 시간에 미친 영향
- proof audit가 발견한 오류와 후속 재사용 횟수
- workshop 전후 분야 사이 citation·공동 연구 network 변화
- mentoring과 software·dataset 기여의 장기 효과
- 지위와 발언권이 검증 과정에 만드는 편향

핵심은 하나의 새 점수로 theorem-credit을 대체하는 것이 아니라 다양한 기여가 보이도록 여러 증거를 함께 기록하는 것이다.